In [1]:
%pip -q install ultralytics==8.4.1 lap==0.5.12 pycocotools==2.0.11 tabulate
%pip -q install --no-deps trackeval==1.3.0
!git clone https://github.com/mdmohsin212/WasteScope.git

Cloning into 'WasteScope'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (240/240), done.
remote: Total 434 (delta 226), reused 362 (delta 160), pack-reused 0 (from 0)
Receiving objects: 100% (434/434), 27.34 MiB | 30.13 MiB/s, done.
Resolving deltas: 100% (226/226), done.


In [2]:
from google.colab import drive
import sys
from pathlib import Path
drive.mount("/content/drive")

REPO_DIR = Path("/content/WasteScope")
sys.path.insert(0, str(REPO_DIR))

Mounted at /content/drive


In [3]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path
from types import SimpleNamespace
from zipfile import ZipFile

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.engine.results import Boxes
from ultralytics.trackers.byte_tracker import BYTETracker
from src.evaluation.tracking import evaluate_tracking

In [4]:
BASE = Path("/content/drive/MyDrive/WasteScope")
PREPARED = BASE / "prepared/tracking"
DETECTOR = "D1_yolo26n_960"
RUN_NAME = "T0_bytetrack"
SPLIT = "val"

WORK = Path("/content/wastescope_tracking")
CACHE = BASE / "runs/tracking/detection_cache" / DETECTOR / f"{SPLIT}_960"
REPORT = BASE / "reports/tracking" / RUN_NAME / SPLIT

for folder in (WORK, CACHE, REPORT):
    folder.mkdir(parents=True, exist_ok=True)

data_config = json.loads((PREPARED / "data_config.json").read_text())

frames = pd.read_csv(PREPARED / data_config["frames_file"], dtype={"Sequence": str})
annotations = pd.read_csv(PREPARED / data_config["annotations_file"], dtype={"Sequence": str})

frames = frames[frames["Split"].eq(SPLIT)].sort_values(["Sequence", "Frame"])
annotations = annotations[annotations["Split"].eq(SPLIT)]

print("Sequences:", frames["Sequence"].nunique())
print("Frames:", len(frames), "| GT instances:", len(annotations))

Sequences: 13
Frames: 1480 | GT instances: 5698


In [5]:
cache_csv = CACHE / "detections.csv"
cache_json = CACHE / "metadata.json"
columns = ["Sequence", "Frame", "x1", "y1", "x2", "y2", "Score", "ClassID"]

if cache_csv.exists() and cache_json.exists():
    detections = pd.read_csv(cache_csv, dtype={"Sequence": str})
    cache_metadata = json.loads(cache_json.read_text())
else:
    local_archive = WORK / f"{SPLIT}.zip"
    shutil.copy2(PREPARED / f"{SPLIT}.zip", local_archive)
    
    weights = WORK / "best.pt"
    shutil.copy2(BASE / "runs/detect" / DETECTOR / "weights/best.pt", weights)
    model = YOLO(str(weights))
    
    rows = []
    started = time.perf_counter()
    with ZipFile(local_archive) as archive:
        for start in tqdm(range(0, len(frames), 16), desc="Caching detections"):
            batch = frames.iloc[start:start + 16]
            images = [
                cv2.imdecode(
                    np.frombuffer(archive.read(frame.ImageMember), np.uint8),
                    cv2.IMREAD_COLOR
                )
                for frame in batch.itertuples(index=False)
            ]
            
            results = model.predict(
                images,
                imgsz=960,
                batch=16,
                device=0,
                half=True,
                conf=0.001, 
                max_det=300,
                nms=False, 
                rect=True, 
                augment=False, 
                verbose=False
            )
            for frame, result in zip(batch.itertuples(index=False), results):
                rows.extend(
                    [frame.Sequence, frame.Frame, *box]
                    for box in result.boxes.data.cpu().numpy()
                )
    
    detections = pd.DataFrame(rows, columns=columns)
    cache_metadata = {
        "detector": DETECTOR,
        "checkpoint_sha256": hashlib.sha256(weights.read_bytes()).hexdigest(),
        "split": SPLIT, 
        "frames": len(frames),
        "imgsz": 960, 
        "confidence": 0.001, 
        "max_det": 300,
        "half": True, 
        "rect": True, 
        "nms": False,
        "gpu": torch.cuda.get_device_name(0),
        "ultralytics": version("ultralytics"),
        "cache_generation_seconds": time.perf_counter() - started
    }
    temp = cache_csv.with_suffix(".tmp")
    detections.to_csv(temp, index=False)
    temp.replace(cache_csv)
    cache_json.write_text(json.dumps(cache_metadata, indent=2))

print("Cached detection rows:", len(detections))

Caching detections:   0%|          | 0/93 [00:00<?, ?it/s]

Cached detection rows: 11340


In [6]:
default_yaml = Path(ultralytics.__file__).parent / "cfg/trackers/bytetrack.yaml"

tracker_config = yaml.safe_load(default_yaml.read_text())
detection_groups = {
    key: group[columns[2:]].to_numpy(dtype=np.float32)
    for key, group in detections.groupby(["Sequence", "Frame"])
}

display(pd.Series(tracker_config, name="T0 defaults"))

,T0 defaults
tracker_type,bytetrack
track_high_thresh,0.25
track_low_thresh,0.1
new_track_thresh,0.25
track_buffer,30
match_thresh,0.8
fuse_score,True


In [7]:
track_rows = []
started = time.perf_counter()

for sequence, sequence_frames in tqdm(
    frames.groupby("Sequence", sort=True), desc="ByteTrack"
):
    tracker = BYTETracker(SimpleNamespace(**tracker_config), frame_rate=30)
    
    for frame in sequence_frames.itertuples(index=False):
        data = detection_groups.get(
            (sequence, frame.Frame), np.empty((0, 6), dtype=np.float32)
        )
        
        boxes = Boxes(data, orig_shape=(frame.Height, frame.Width))
        
        for result in tracker.update(boxes):
            x1, y1, x2, y2, track_id, score, class_id, _ = result
            x1, y1, x2, y2 = np.clip(
                [x1, y1, x2, y2],
                [0, 0, 0, 0],
                [frame.Width, frame.Height, frame.Width, frame.Height]
            )
            track_rows.append([
                sequence, frame.Frame, int(track_id), int(class_id),
                x1, y1, x2, y2, float(score)
            ])
            

association_seconds = time.perf_counter() - started
tracks = pd.DataFrame(track_rows, columns=[
    "Sequence", "Frame", "TrackID", "ClassID",
    "x1", "y1", "x2", "y2", "Score"
])
tracks.to_csv(REPORT / "tracks.csv", index=False)

ByteTrack:   0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
overall, per_sequence = evaluate_tracking(frames, annotations, tracks)
overall.insert(0, "Split", SPLIT)
overall.insert(0, "Run", RUN_NAME)

overall.to_csv(REPORT / "overall.csv", index=False)
per_sequence.to_csv(REPORT / "per_sequence.csv")
(REPORT / "tracker.yaml").write_text(
    yaml.safe_dump(tracker_config, sort_keys=False)
)

configuration = {
    "status": "complete",
    "run": RUN_NAME, "split": SPLIT,
    "sequences": int(frames["Sequence"].nunique()),
    "frames": len(frames),
    "detector": cache_metadata,
    "tracker": tracker_config,
    "tracker_frame_rate_argument": 30,
    "annotations_file": data_config["annotations_file"],
    "annotations_sha256": hashlib.sha256(
        (PREPARED / data_config["annotations_file"]).read_bytes()
    ).hexdigest(),
    "manifest_sha256": data_config["manifest_sha256"],
    "evaluation": {
        "class_aware": True,
        "HOTA_IoU": "0.05:0.05:0.95",
        "IDF1_MOTA_IoU": 0.5,
        "counting": "All unique activated track IDs per sequence",
        "duration_filter": None,
    },
    "association_seconds": association_seconds,
    "versions": {
        package: version(package)
        for package in ("ultralytics", "trackeval", "numpy", "scipy", "pandas", "lap")
    },
}
(REPORT / "evaluation_config.json").write_text(
    json.dumps(configuration, indent=2)
)

(REPORT / "README.md").write_text(
    f"# {RUN_NAME} — {SPLIT}\n\n"
    f"Detector: {DETECTOR}, imgsz=960. Default ByteTrack configuration.\n\n"
    "HOTA, DetA, AssA, IDF1, MOTA and MAPE are percentages.\n"
    "Matching is class-aware. Counting includes every activated track ID.\n\n"
    + overall.round(4).to_markdown(index=False)
    + "\n\n## Per-sequence results\n\n"
    + per_sequence.round(4).to_markdown()
    + "\n",
    encoding="utf-8",
)

display(overall.round(4))
display(per_sequence.round(4))
print("T0 report saved:", REPORT)

/usr/local/lib/python3.13/dist-packages/trackeval/metrics/hota.py:72: RuntimeWarning: invalid value encountered in divide
  global_alignment_score = potential_matches_count / (gt_id_count + tracker_id_count - potential_matches_count)
/usr/local/lib/python3.13/dist-packages/trackeval/metrics/hota.py:72: RuntimeWarning: invalid value encountered in divide
  global_alignment_score = potential_matches_count / (gt_id_count + tracker_id_count - potential_matches_count)
/usr/local/lib/python3.13/dist-packages/trackeval/metrics/hota.py:72: RuntimeWarning: invalid value encountered in divide
  global_alignment_score = potential_matches_count / (gt_id_count + tracker_id_count - potential_matches_count)
/usr/local/lib/python3.13/dist-packages/trackeval/metrics/hota.py:72: RuntimeWarning: invalid value encountered in divide
  global_alignment_score = potential_matches_count / (gt_id_count + tracker_id_count - potential_matches_count)
/usr/local/lib/python3.13/dist-packages/trackeval/metrics/hota.p

,Run,Split,HOTA,DetA,AssA,IDF1,MOTA,IDSW,GT,Predicted,MAE,RMSE,SAD,MAPE
0,T0_bytetrack,val,37.321,15.2075,94.7368,28.2051,11.1111,0,56,102,3.6923,5.3349,48,92.2253


,HOTA,DetA,AssA,IDF1,MOTA,IDSW,GT,Predicted,CountError,AbsCountError
Sequence,,,,,,,,,,
0010_1,89.4737,89.4737,89.4737,100.0000,100.0000,0,1,1,0,0
0021_1,0.0000,0.0000,0.0000,0.0000,0.0000,0,1,0,-1,1
0024_1,50.1683,35.9649,73.6842,40.0000,0.0000,0,8,21,13,13
0029_1,38.3417,20.8772,73.6842,28.5714,-25.0000,0,6,12,6,6
0039_1,81.8551,71.5038,94.7368,88.8889,80.0000,0,5,8,3,3
0063_1,0.0000,0.0000,0.0000,0.0000,0.0000,0,7,12,5,5
0069_1,78.9474,78.9474,78.9474,100.0000,100.0000,0,1,1,0,0
0075_1,0.0000,0.0000,0.0000,0.0000,-33.3333,0,5,9,4,4
0094_1,73.6842,73.6842,73.6842,100.0000,100.0000,0,2,3,1,1


T0 report saved: /content/drive/MyDrive/WasteScope/reports/tracking/T0_bytetrack/val
